# Semantic Faithfulness and Entropy Production: Complete Pipeline Demo

This notebook demonstrates the complete pipeline for measuring **Semantic Faithfulness ($\mathcal{F}_S$)** and **Semantic Entropy Production (SEP)** for Large Language Model outputs.

## 📖 Overview

The Semantic Divergence Metrics (SDM) framework provides information-theoretically principled measures of how faithfully an LLM's answer represents the information in a provided context.

**Key Concepts:**
- **Semantic Faithfulness ($\mathcal{F}_S$)**: Measures how well the answer aligns with the optimal information channel from context to question
- **Semantic Entropy Production (SEP)**: Measures the irreversibility in the question-answering process
- **UDIB Clustering**: Automatic discovery of semantic topics from text

**Pipeline Steps:**
1. Tokenize text into sentences
2. Generate embeddings for each sentence (with caching)
3. Cluster sentences into semantic topics using UDIB (with caching)
4. Compute probability distributions over topics
5. Calculate Semantic Faithfulness and Entropy Production metrics

**⚡ Performance Note:** This notebook uses intelligent caching to avoid re-computing expensive operations like embeddings and clustering when run multiple times.

---

## 1. Installation and Imports

First, let's import the required packages and set up caching utilities.

In [ ]:
# Uncomment to install the package
# !pip install -e .

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import entropy
import os
import pickle
import hashlib
from pathlib import Path

# SDM package imports
from sdm_package import SemanticMutualInformationAnalyzer, compute_semantic_faithfulness
from sdm_package.DIB_with_KL_upper_bound import DIBAnalyzer

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Create cache directory
CACHE_DIR = Path("demo_cache")
CACHE_DIR.mkdir(exist_ok=True)

print("✓ All imports successful")
print(f"✓ Cache directory: {CACHE_DIR.absolute()}")

### 1.1 Caching Utilities

These functions enable efficient caching of embeddings and clustering results.

In [ ]:
def get_cache_key(data, prefix=""):
    """Generate a cache key from data using hash."""
    if isinstance(data, str):
        content = data
    elif isinstance(data, list):
        content = "|".join(str(item) for item in data)
    else:
        content = str(data)
    
    hash_obj = hashlib.md5(content.encode())
    return f"{prefix}_{hash_obj.hexdigest()[:12]}"

def save_cache(data, cache_key):
    """Save data to cache."""
    cache_path = CACHE_DIR / f"{cache_key}.pkl"
    with open(cache_path, 'wb') as f:
        pickle.dump(data, f)
    print(f"💾 Saved to cache: {cache_key}")

def load_cache(cache_key):
    """Load data from cache if it exists."""
    cache_path = CACHE_DIR / f"{cache_key}.pkl"
    if cache_path.exists():
        with open(cache_path, 'rb') as f:
            data = pickle.load(f)
        print(f"⚡ Loaded from cache: {cache_key}")
        return data
    return None

print("✓ Caching utilities ready")

## 2. Prepare Example Data: Question-Context-Answer (QCA) Triplet

We'll use a realistic example based on NVIDIA's business risks to demonstrate the pipeline.

In [ ]:
# Question: Comprehensive multi-topic question
question = """
Provide a comprehensive overview of NVIDIA's business risks, including:
1. Supply chain and manufacturing dependencies
2. Competition and market position challenges
3. Technological and product development risks
4. Regulatory and geopolitical concerns
"""

# Context: Excerpt from NVIDIA 10-K Risk Factors
context = """
NVIDIA faces significant risks from supply chain dependencies on third-party manufacturers,
particularly in Taiwan and other regions subject to geopolitical tensions. 
Any disruption in semiconductor fabrication could severely impact our ability to meet demand.

The competitive landscape in AI accelerators and GPUs is intensifying. 
Major technology companies are developing custom silicon solutions, 
and established semiconductor firms are investing heavily in competing architectures.
Our market share and pricing power could be negatively affected.

Rapid technological change requires continuous innovation in chip architecture, 
software ecosystems, and AI capabilities. Product development cycles are lengthy and expensive,
with no guarantee of market acceptance. 
Technology transitions risk rendering existing inventory obsolete.

Export controls and sanctions targeting advanced semiconductors, particularly regarding China,
create uncertainty and limit our addressable market. 
Regulatory changes in data privacy, AI safety, and antitrust enforcement could impose 
additional compliance costs and business restrictions.
"""

# Answer: LLM-generated response
answer = """
NVIDIA faces several critical business risks across multiple dimensions:

1. Supply Chain Vulnerabilities: The company's heavy reliance on Taiwan-based semiconductor 
manufacturers creates significant geopolitical exposure. Any disruption in fabrication capacity
could severely constrain NVIDIA's ability to meet market demand.

2. Intensifying Competition: The AI accelerator market is becoming increasingly competitive,
with hyperscalers developing custom chips and traditional competitors investing in alternatives.
This threatens NVIDIA's dominant market position and pricing power.

3. Technology and Product Risks: Rapid innovation cycles demand continuous R&D investment
with uncertain returns. Product development is expensive and lengthy, with risks of 
obsolescence and market rejection.

4. Regulatory and Geopolitical Headwinds: Export controls limiting sales to China restrict
addressable markets, while evolving AI regulations and antitrust scrutiny create compliance
burdens and potential business constraints.
"""

print(f"Question length: {len(question)} characters")
print(f"Context length: {len(context)} characters")
print(f"Answer length: {len(answer)} characters")
print("\n✓ QCA triplet prepared")

## 3. Pipeline Step 1: Sentence Tokenization

The first step is to break down each text into sentences using NLTK's sentence tokenizer.

In [ ]:
import nltk
from nltk.tokenize import sent_tokenize

# Download punkt tokenizer if needed
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

# Tokenize into sentences
question_sentences = sent_tokenize(question.strip())
context_sentences = sent_tokenize(context.strip())
answer_sentences = sent_tokenize(answer.strip())

print("Question Sentences:")
for i, sent in enumerate(question_sentences, 1):
    print(f"  {i}. {sent[:80]}..." if len(sent) > 80 else f"  {i}. {sent}")

print(f"\nContext Sentences: {len(context_sentences)} sentences")
print(f"Answer Sentences: {len(answer_sentences)} sentences")
print(f"\nTotal sentences to analyze: {len(question_sentences) + len(context_sentences) + len(answer_sentences)}")

# Combine all sentences for embedding
all_sentences = question_sentences + context_sentences + answer_sentences

## 4. Pipeline Step 2: Sentence Embedding (with Caching)

Each sentence is encoded into a dense vector representation. **Embeddings are cached** to avoid re-computation on subsequent runs.

In [ ]:
from sentence_transformers import SentenceTransformer

# Generate cache key based on sentences
embedding_cache_key = get_cache_key(all_sentences, "embeddings")

# Try to load from cache
embeddings = load_cache(embedding_cache_key)

if embeddings is None:
    # Cache miss - generate embeddings
    print("\n🔄 Generating embeddings (this may take a moment)...")
    embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    print(f"✓ Loaded embedding model: all-MiniLM-L6-v2")
    print(f"  Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")
    
    embeddings = embedding_model.encode(all_sentences, show_progress_bar=True)
    
    # Save to cache for future runs
    save_cache(embeddings, embedding_cache_key)
else:
    print("  (Skipped embedding generation - using cached results)")

print(f"\n✓ Embeddings ready: shape {embeddings.shape}")
print(f"  ({embeddings.shape[0]} sentences × {embeddings.shape[1]} dimensions)")

## 5. Pipeline Step 3: Topic Discovery via UDIB Clustering (with Caching)

The **Upper-Bounded Deterministic Information Bottleneck (UDIB)** algorithm automatically discovers semantic topics. **Clustering results are cached** for efficiency.

### Information Bottleneck Objective

UDIB minimizes:
$$I(X; T) - \beta \cdot I(T; Y)$$

subject to: $I(X; T) \leq I_{max}$

Where:
- $X$ = sentence embeddings
- $T$ = topic assignments (clusters)
- $Y$ = text identity (Question/Context/Answer)
- $\beta$ = trade-off parameter
- $I_{max}$ = upper bound on compression

In [ ]:
# Define clustering parameters
tau_values = np.logspace(-2, 2, 30)
max_n_clusters = 15
seed = 42

# Generate cache key based on embeddings and parameters
clustering_params = f"{embedding_cache_key}_tau{len(tau_values)}_maxc{max_n_clusters}_seed{seed}"
clustering_cache_key = get_cache_key(clustering_params, "clustering")

# Try to load from cache
cached_clustering = load_cache(clustering_cache_key)

if cached_clustering is None:
    # Cache miss - run UDIB clustering
    print("\n🔄 Running UDIB clustering (this may take a minute)...")
    print(f"  Tau range: [{tau_values.min():.4f}, {tau_values.max():.4f}]")
    print(f"  Number of values: {len(tau_values)}")
    
    dib_analyzer = DIBAnalyzer(embeddings, all_sentences)
    dib_analyzer.run(tau_values, max_n_clusters=max_n_clusters, seed=seed)
    
    # Get recommendation
    recommendation, _ = dib_analyzer.get_recommendation(min_clusters=3, metric='kink_angle')
    
    # Cache both the analyzer and recommendation
    cached_clustering = {
        'recommendation': recommendation,
        'dib_analyzer': dib_analyzer
    }
    save_cache(cached_clustering, clustering_cache_key)
    
else:
    print("  (Skipped clustering - using cached results)")
    recommendation = cached_clustering['recommendation']
    dib_analyzer = cached_clustering['dib_analyzer']

print(f"\n✓ UDIB clustering complete")
print(f"  Recommended clusters: {recommendation['n_clusters']}")
print(f"  Kink angle (robustness): {recommendation['kink_angle']:.2f}°")
print(f"  Stable tau range: [{recommendation['tau_min']:.4f}, {recommendation['tau_max']:.4f}]")
print(f"  Cluster entropy H(c): {recommendation['H(c)']:.3f} bits")

### 5.1 Visualize Clustering Results

The UDIB algorithm identifies "kinks" in the information curve where stable clustering solutions exist.

In [ ]:
# Plot clustering diagnostics
dib_analyzer.plot(recommendation, metric='kink_angle', window_size=2)

### 5.2 Analyze Discovered Topics

Let's examine what semantic topics were discovered by the UDIB algorithm.

In [ ]:
# Analyze cluster topics
topic_analysis = dib_analyzer.analyze_cluster_topics(
    recommendation, 
    top_n_words=5, 
    num_example_sentences=2
)

## 6. Pipeline Step 4: Compute Probability Distributions

For each text (Question, Context, Answer), we compute the probability distribution over the discovered topics.

$$p_j^{(\text{text})} = \frac{\text{# sentences in topic } j}{\text{total sentences in text}}$$

In [ ]:
# Get cluster assignments
assignments = recommendation['assignments']
n_topics = recommendation['n_clusters']

# Split assignments back into Q, C, A
n_q = len(question_sentences)
n_c = len(context_sentences)
n_a = len(answer_sentences)

assignments_q = assignments[:n_q]
assignments_c = assignments[n_q:n_q+n_c]
assignments_a = assignments[n_q+n_c:]

# Compute probability distributions
def compute_distribution(assignments, n_topics):
    counts = np.bincount(assignments, minlength=n_topics)
    return counts / counts.sum()

p_question = compute_distribution(assignments_q, n_topics)
p_context = compute_distribution(assignments_c, n_topics)
p_answer = compute_distribution(assignments_a, n_topics)

print(f"Probability distributions computed:")
print(f"  p_question shape: {p_question.shape}")
print(f"  p_context shape: {p_context.shape}")
print(f"  p_answer shape: {p_answer.shape}")
print(f"\n  Sum checks (should be 1.0):")
print(f"    p_question.sum() = {p_question.sum():.6f}")
print(f"    p_context.sum() = {p_context.sum():.6f}")
print(f"    p_answer.sum() = {p_answer.sum():.6f}")

### 6.1 Visualize Probability Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

topics = np.arange(n_topics)

axes[0].bar(topics, p_question, color='steelblue', alpha=0.7)
axes[0].set_title('Question Distribution p(Q)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Topic')
axes[0].set_ylabel('Probability')
axes[0].set_xticks(topics)
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(topics, p_context, color='forestgreen', alpha=0.7)
axes[1].set_title('Context Distribution p(C)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Topic')
axes[1].set_ylabel('Probability')
axes[1].set_xticks(topics)
axes[1].grid(axis='y', alpha=0.3)

axes[2].bar(topics, p_answer, color='coral', alpha=0.7)
axes[2].set_title('Answer Distribution p(A)', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Topic')
axes[2].set_ylabel('Probability')
axes[2].set_xticks(topics)
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Compute entropies
H_Q = entropy(p_question, base=2)
H_C = entropy(p_context, base=2)
H_A = entropy(p_answer, base=2)

print(f"\nEntropies:")
print(f"  H(Q) = {H_Q:.3f} bits")
print(f"  H(C) = {H_C:.3f} bits")
print(f"  H(A) = {H_A:.3f} bits")

## 7. Pipeline Step 5: Compute Semantic Faithfulness

### 7.1 Theoretical Framework

Model LLM question-answering as information flow through transition matrices:

**Goal Channel (Q-matrix)**: $P(\text{topic } j \text{ in Q} | \text{topic } i \text{ in C})$

**Actual Channel (A-matrix)**: $P(\text{topic } j \text{ in A} | \text{topic } i \text{ in C})$

Both must satisfy:
1. Row-stochastic: $\sum_j Q_{ij} = 1$, $\sum_j A_{ij} = 1$
2. Marginal constraints:
   - $p^{(q)} = p^{(c)T} \cdot Q$
   - $p^{(a)} = p^{(c)T} \cdot A$

### 7.2 Optimal Goal Channel

The optimal $Q^*$ minimizes KL divergence from the actual answer channel:

$$Q^* = \arg\min_Q \text{KL}(A \| Q) = \sum_{i,j} p_i^{(c)} A_{ij} \log\frac{A_{ij}}{Q_{ij}}$$

### 7.3 Semantic Faithfulness Metric

$$\mathcal{F}_S = \frac{1}{1 + D_{\min}}$$

where $D_{\min} = \text{KL}(A \| Q^*)$

Properties:
- Range: $\mathcal{F}_S \in (0, 1]$
- $\mathcal{F}_S = 1$ ⟹ Perfect faithfulness ($A = Q^*$)
- $\mathcal{F}_S \to 0$ ⟹ Low faithfulness (high divergence)

In [ ]:
# Compute Semantic Faithfulness using Csiszár-Tusnády alternating minimization
results = compute_semantic_faithfulness(
    p_c=p_context,
    p_q=p_question,
    p_a=p_answer,
    tol_outer=1e-7,
    max_outer_iter=100,
    debug=True  # Show convergence details
)

## 8. Pipeline Step 6: Compute Semantic Entropy Production

### 8.1 Thermodynamic Interpretation

View the LLM as a **bipartite information engine**:
- Sub-system X: Observable (context → answer)
- Sub-system Y: Hidden controller (Maxwell's demon)

### 8.2 System Entropy Production

Measures semantic expansion/compression:

$$\dot{S}_{\text{system}} = H(A) - H(C)$$

- $\dot{S}_{\text{system}} > 0$: Semantic expansion (LLM elaborates)
- $\dot{S}_{\text{system}} < 0$: Semantic compression (LLM summarizes)
- $\dot{S}_{\text{system}} = 0$: Semantic conservation

### 8.3 Total Entropy Production

Measures irreversibility of information flow:

$$\dot{S}_{\text{total}} \approx \frac{1}{\mathcal{F}_S} - 1 = D_{\min}$$

This establishes the **inverse relationship**:
- High $\mathcal{F}_S$ → Low $\dot{S}_{\text{total}}$ (faithful answers)
- Low $\mathcal{F}_S$ → High $\dot{S}_{\text{total}}$ (unfaithful answers)

In [ ]:
# Extract results
F_S = results['F_S']
D_min = results['D_min']
A_star = results['A_star']
Q_star = results['Q_star']
converged = results['converged']
iterations = results['iterations']

# Compute entropy production metrics
SEP_system = H_A - H_C  # System entropy production
SEP_total = D_min  # Total entropy production ≈ 1/F_S - 1

# Theoretical approximation
SEP_total_approx = 1/F_S - 1

print("="*80)
print("FINAL RESULTS")
print("="*80)
print(f"\n1. Semantic Faithfulness:")
print(f"   𝓕_S = {F_S:.6f}")
print(f"   D_min = {D_min:.6f} bits")
print(f"   Converged: {converged} (iterations: {iterations})")

print(f"\n2. Entropy Production:")
print(f"   SEP_system = H(A) - H(C) = {SEP_system:.6f} bits")
if SEP_system > 0:
    print(f"   → Semantic EXPANSION (answer elaborates on context)")
elif SEP_system < 0:
    print(f"   → Semantic COMPRESSION (answer summarizes context)")
else:
    print(f"   → Semantic CONSERVATION")

print(f"\n   SEP_total = {SEP_total:.6f} bits")
print(f"   SEP_total ≈ 1/𝓕_S - 1 = {SEP_total_approx:.6f} bits")
print(f"   Approximation error: {abs(SEP_total - SEP_total_approx)/SEP_total*100:.2f}%")

print(f"\n3. Interpretation:")
if F_S > 0.85:
    print(f"   ✓ HIGH faithfulness - answer closely aligns with question")
elif F_S > 0.65:
    print(f"   ○ MODERATE faithfulness - reasonable alignment")
else:
    print(f"   ✗ LOW faithfulness - significant divergence from question")

if SEP_total < 0.1:
    print(f"   ✓ Very LOW entropy production - near-optimal information flow")
elif SEP_total < 0.3:
    print(f"   ○ LOW entropy production - good information efficiency")
elif SEP_total < 0.5:
    print(f"   △ MODERATE entropy production")
else:
    print(f"   ✗ HIGH entropy production - inefficient/hallucination risk")

print("="*80)

## 9. Visualize Transition Matrices

Let's examine the learned transition matrices $Q^*$ (goal channel) and $A^*$ (actual channel).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot Q* (Goal Channel)
im1 = axes[0].imshow(Q_star, cmap='Blues', aspect='auto', interpolation='nearest')
axes[0].set_title('Q* (Goal Channel): Context → Question', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Question Topics')
axes[0].set_ylabel('Context Topics')
plt.colorbar(im1, ax=axes[0])

# Plot A* (Actual Channel)
im2 = axes[1].imshow(A_star, cmap='Oranges', aspect='auto', interpolation='nearest')
axes[1].set_title('A* (Actual Channel): Context → Answer', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Answer Topics')
axes[1].set_ylabel('Context Topics')
plt.colorbar(im2, ax=axes[1])
# Plot divergence: A - Q (pointwise)
divergence = A_star - Q_star
max_abs = np.abs(divergence).max()
im3 = axes[2].imshow(divergence, cmap='RdBu_r', aspect='auto', 
                      interpolation='nearest', vmin=-max_abs, vmax=max_abs)
axes[2].set_title('A* - Q* (Divergence)', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Topics')
axes[2].set_ylabel('Context Topics')
plt.colorbar(im3, ax=axes[2])

plt.tight_layout()
plt.show()

print("\nMatrix properties:")
print(f"  Q* row sums (should be 1): min={Q_star.sum(axis=1).min():.6f}, max={Q_star.sum(axis=1).max():.6f}")
print(f"  A* row sums (should be 1): min={A_star.sum(axis=1).min():.6f}, max={A_star.sum(axis=1).max():.6f}")
print(f"\n  Marginal constraint check:")
print(f"    ||p_c^T Q* - p_q||: {np.linalg.norm(p_context @ Q_star - p_question):.6e}")
print(f"    ||p_c^T A* - p_a||: {np.linalg.norm(p_context @ A_star - p_answer):.6e}")

## 10. Cache Management

View cached files and optionally clear the cache.

In [ ]:
# List cached files
cache_files = list(CACHE_DIR.glob("*.pkl"))
print(f"Cached files ({len(cache_files)} total):")
for f in cache_files:
    size_kb = f.stat().st_size / 1024
    print(f"  - {f.name} ({size_kb:.1f} KB)")

print(f"\nTotal cache size: {sum(f.stat().st_size for f in cache_files) / 1024:.1f} KB")
print("\nTo clear cache, uncomment and run: !rm -rf demo_cache/")

## 11. Summary and Key Takeaways

### Pipeline Overview

1. **Sentence Tokenization** → Break text into sentences
2. **Embedding Generation** → Convert sentences to dense vectors (cached)
3. **UDIB Clustering** → Discover semantic topics automatically (cached)
4. **Distribution Computation** → Calculate topic probabilities
5. **Faithfulness Calculation** → Measure alignment via $\mathcal{F}_S$
6. **Entropy Production** → Quantify irreversibility via SEP

### Performance Optimizations

This notebook implements **intelligent caching** for expensive operations:
- ⚡ **Embeddings**: Cached based on sentence content hash
- ⚡ **Clustering**: Cached based on embeddings + parameters
- 🚀 **Speedup**: ~10-100x faster on subsequent runs

### Key Metrics

- **$\mathcal{F}_S$ (Semantic Faithfulness)**: Higher is better (0 < $\mathcal{F}_S$ ≤ 1)
  - $\mathcal{F}_S$ > 0.85: High faithfulness
  - 0.65 < $\mathcal{F}_S$ < 0.85: Moderate faithfulness
  - $\mathcal{F}_S$ < 0.65: Low faithfulness

- **SEP (Semantic Entropy Production)**: Lower is better
  - SEP < 0.1 bits: Very low (near-optimal)
  - 0.1-0.3 bits: Low (good)
  - 0.3-0.5 bits: Moderate
  - SEP > 0.5 bits: High (potential hallucination)

### Inverse Relationship

$$\dot{S}_{\text{total}} \approx \frac{1}{\mathcal{F}_S} - 1$$

This fundamental relationship connects information-theoretic faithfulness with thermodynamic irreversibility.

---

## 📚 References

1. **Halperin, I. (2025).** "Information-Theoretic Faithfulness Metrics for Large Language Models." *To be published.*

2. **Halperin, I. (2025).** "Prompt-Response Semantic Divergence Metrics for Faithfulness Hallucination Detection in Large Language Models."
   - arXiv: [2508.10192](https://arxiv.org/abs/2508.10192)

3. **Halperin, I. (2025).** "Topic Identification in LLM Input-Output Pairs through the Lens of Information Bottleneck."
   - arXiv: [2509.03533](https://arxiv.org/abs/2509.03533)

---

**Repository**: https://github.com/ighalp/semantic-faithfulness-sdm

**License**: MIT